# MindSpore ResNet18 on CIFAR-10

这份 notebook 对应 README 里的 4 个作业要求：

1. 按 `resnet50` 模板改写出 `resnet18`
2. 训练并保存 `cifar10resnet18.ckpt`，同时画出 loss / accuracy 曲线
3. 可视化训练后 ResNet18 第一层 `64` 个 `7×7` 卷积核
4. 模仿 AlexNet 2012 Fig.4，展示 5 张测试图像及其在最后隐藏层特征空间中的 6 个最近邻训练图像

运行建议：

- 直接顺序执行全部单元格
- 首次运行会自动下载 CIFAR-10 二进制数据集
- 推荐在带 GPU 的在线 notebook 中运行
- 跑完后，`outputs/` 目录中的图片可以直接用于制作 `report.pdf`

## 环境说明

这份 notebook 假设环境中已经安装：

- `mindspore`
- `matplotlib`
- `numpy`

如果在线环境没有安装 MindSpore，请先按你所用平台的方式安装后再运行后续单元格。

## 依赖检测与自动安装

先运行下面这个单元格。它会针对 `Python 3.7.10` 做兼容处理：

- 避免用直接 `import matplotlib` 的方式做依赖检测
- 避免 `modelarts-sdk` 中同名包污染 `matplotlib/packaging`
- 在 `Python 3.7` 下优先安装更稳的旧版本依赖

如果这个单元格安装了新库，请重启 kernel，然后从后面的导入单元格继续重新执行。


In [ ]:
# Python 3.7 compatible package setup for this notebook
import importlib.util
import subprocess
import sys

PY37 = sys.version_info[:2] == (3, 7)
print(f"Python version: {sys.version.split()[0]}")

# ModelArts may inject a shadowed matplotlib/packaging path that breaks on Python 3.7.
removed_paths = [p for p in list(sys.path) if "modelarts-sdk" in p]
if removed_paths:
    sys.path[:] = [p for p in sys.path if "modelarts-sdk" not in p]
    print("Removed shadowed paths from sys.path:")
    for p in removed_paths:
        print(f"  - {p}")

if PY37:
    REQUIRED_PACKAGES = {
        "numpy": "numpy<=1.21.6",
        "packaging": "packaging<22",
        "matplotlib": "matplotlib<3.6",
        "mindspore": "mindspore==2.4.10",
    }
else:
    REQUIRED_PACKAGES = {
        "numpy": "numpy",
        "packaging": "packaging",
        "matplotlib": "matplotlib",
        "mindspore": "mindspore",
    }


def module_available(module_name):
    return importlib.util.find_spec(module_name) is not None


def ensure_packages(package_map):
    missing = []
    for module_name, package_name in package_map.items():
        try:
            available = module_available(module_name)
        except Exception as exc:
            print(f"[BROKEN] {module_name}: {exc}")
            available = False

        if available:
            print(f"[OK] {module_name}")
        else:
            print(f"[MISSING] {module_name} -> will install `{package_name}`")
            missing.append(package_name)

    if not missing:
        print("All required packages are already installed.")
        return False

    if PY37:
        print("Python 3.7 detected. Installing Python 3.7 compatible package versions.")

    for package_name in missing:
        print(f"Installing {package_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

    print("Installation finished.")
    print("Please restart the kernel, then rerun the notebook from the device-check cell below.")
    return True


packages_changed = ensure_packages(REQUIRED_PACKAGES)


## 运行设备检测

先运行下面这个单元格。它会优先尝试使用 `GPU`，如果当前 MindSpore 或运行环境不支持 GPU，就会明确回退到 `CPU`，并打印实际的 `device_target`。

如果这里最终显示的是 `CPU`，那么后面的训练就不会使用 GPU。

In [ ]:
# GPU detection and fallback
import os

IS_KAGGLE = os.path.exists("/kaggle")
print(f"Kaggle environment: {IS_KAGGLE}")

try:
    import mindspore as ms
except ImportError:
    raise RuntimeError("mindspore is not installed yet. Please run the package installation cell first.")


def configure_device():
    last_error = None
    for target in ["GPU", "CPU"]:
        try:
            ms.set_context(mode=ms.GRAPH_MODE, device_target=target)
            current = ms.get_context("device_target")
            print(f"MindSpore device_target: {current}")
            if current == "GPU":
                print("GPU is available and will be used for training.")
            else:
                print("GPU is unavailable. Training will run on CPU.")
            return current
        except Exception as exc:
            print(f"Unable to use {target}: {exc}")
            last_error = exc

    raise RuntimeError(f"Failed to configure device target. Last error: {last_error}")


DEVICE_TARGET = configure_device()

In [ ]:
import sys
import tarfile
import urllib.request
from pathlib import Path

# Remove ModelArts path shadowing again in case the kernel was restarted.
sys.path[:] = [p for p in sys.path if "modelarts-sdk" not in p]

import mindspore as ms
import mindspore.dataset as ds
import mindspore.dataset.transforms as transforms
import mindspore.dataset.vision as vision
import numpy as np
import matplotlib.pyplot as plt
from mindspore import Tensor, load_checkpoint, load_param_into_net, nn, ops, save_checkpoint
from mindspore.train import Model
from mindspore.train.callback import Callback, LossMonitor, TimeMonitor

if "DEVICE_TARGET" not in globals():
    try:
        ms.set_context(mode=ms.GRAPH_MODE, device_target="GPU")
        DEVICE_TARGET = ms.get_context("device_target")
    except Exception:
        ms.set_context(mode=ms.GRAPH_MODE, device_target="CPU")
        DEVICE_TARGET = ms.get_context("device_target")

ms.set_seed(42)
ds.config.set_seed(42)

DATA_ROOT = Path("datasets")
OUTPUT_ROOT = Path("outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CIFAR10_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz"
CIFAR10_ARCHIVE = DATA_ROOT / "cifar-10-binary.tar.gz"
CIFAR10_DIR = DATA_ROOT / "cifar-10-batches-bin"
CKPT_PATH = Path("cifar10resnet18.ckpt")

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

MEAN = np.array([0.4914, 0.4822, 0.4465], dtype=np.float32)
STD = np.array([0.2023, 0.1994, 0.2010], dtype=np.float32)

BATCH_SIZE = 128
EPOCHS = 30
MAX_LR = 0.05
MIN_LR = 1e-4
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
TARGET_ACCURACY = 0.85

print(f"MindSpore version: {ms.__version__}")
print(f"Device target: {DEVICE_TARGET}")
print(f"Working directory: {Path.cwd()}")


In [ ]:
def download_and_extract_cifar10():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)

    if CIFAR10_DIR.exists():
        print(f"CIFAR-10 already exists: {CIFAR10_DIR}")
        return

    if not CIFAR10_ARCHIVE.exists():
        print(f"Downloading CIFAR-10 from: {CIFAR10_URL}")
        urllib.request.urlretrieve(CIFAR10_URL, CIFAR10_ARCHIVE)
        print(f"Archive saved to: {CIFAR10_ARCHIVE}")

    print("Extracting CIFAR-10 archive...")
    with tarfile.open(CIFAR10_ARCHIVE, "r:gz") as tar:
        tar.extractall(DATA_ROOT)
    print(f"Extracted to: {CIFAR10_DIR}")


download_and_extract_cifar10()

In [ ]:
def build_dataset(split="train", batch_size=BATCH_SIZE, shuffle=True, augment=False):
    dataset = ds.Cifar10Dataset(
        dataset_dir=str(CIFAR10_DIR),
        usage=split,
        shuffle=shuffle,
    )

    image_ops = []
    if augment:
        image_ops.append(vision.RandomHorizontalFlip(prob=0.5))
    image_ops.extend([
        vision.Rescale(1.0 / 255.0, 0.0),
        vision.Normalize(mean=MEAN.tolist(), std=STD.tolist()),
        vision.HWC2CHW(),
    ])

    label_ops = transforms.TypeCast(ms.int32)

    dataset = dataset.map(operations=image_ops, input_columns="image")
    dataset = dataset.map(operations=label_ops, input_columns="label")
    dataset = dataset.batch(batch_size, drop_remainder=False)
    return dataset


train_ds = build_dataset(split="train", shuffle=True, augment=True)
train_eval_ds = build_dataset(split="train", shuffle=False, augment=False)
test_ds = build_dataset(split="test", shuffle=False, augment=False)

steps_per_epoch = train_ds.get_dataset_size()
print(f"Train steps per epoch: {steps_per_epoch}")
print(f"Train eval steps: {train_eval_ds.get_dataset_size()}")
print(f"Test steps: {test_ds.get_dataset_size()}")

In [ ]:
def denormalize(images):
    images = np.transpose(images, (0, 2, 3, 1))
    images = images * STD.reshape(1, 1, 1, 3) + MEAN.reshape(1, 1, 1, 3)
    return np.clip(images, 0.0, 1.0)


sample = next(train_ds.create_dict_iterator())
sample_images = sample["image"].asnumpy()
sample_labels = sample["label"].asnumpy()

plt.figure(figsize=(10, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(denormalize(sample_images[i:i + 1])[0])
    plt.title(CLASS_NAMES[int(sample_labels[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 1. 构建 ResNet18

这里保留模板里的核心思想，但把网络结构改成 `ResNet18` 所使用的 `BasicBlock`：

- 第一层卷积仍然是 `64` 个 `7×7` 卷积核
- 残差块使用两层 `3×3` 卷积
- 四个 stage 的 block 数量为 `[2, 2, 2, 2]`
- 最后通过全局平均池化和全连接层输出 10 类结果

In [ ]:
class ConvBNReLU(nn.Cell):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, activate=True):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            pad_mode="pad",
            padding=padding,
            has_bias=False,
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.activate = activate
        self.relu = nn.ReLU()

    def construct(self, x):
        x = self.conv(x)
        x = self.bn(x)
        if self.activate:
            x = self.relu(x)
        return x


class BasicBlock(nn.Cell):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = ConvBNReLU(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.conv2 = ConvBNReLU(out_channels, out_channels, kernel_size=3, stride=1, padding=1, activate=False)
        self.downsample = downsample
        self.relu = nn.ReLU()

    def construct(self, x):
        identity = x

        out = self.conv1(x)
        out = self.conv2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out = out + identity
        out = self.relu(out)
        return out


class ResNet(nn.Cell):
    def __init__(self, block, layers, num_classes=10):
        super().__init__()
        self.in_channels = 64
        self.conv1 = ConvBNReLU(3, 64, kernel_size=7, stride=2, padding=3)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, pad_mode="same")
        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.classifier = nn.Dense(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = ConvBNReLU(
                self.in_channels,
                out_channels * block.expansion,
                kernel_size=1,
                stride=stride,
                padding=0,
                activate=False,
            )

        layers = [block(self.in_channels, out_channels, stride=stride, downsample=downsample)]
        self.in_channels = out_channels * block.expansion

        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.SequentialCell(layers)

    def forward_features(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = ops.mean(x, axis=(2, 3))
        return x

    def construct(self, x):
        x = self.forward_features(x)
        x = self.classifier(x)
        return x


def resnet18(num_classes=10):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)


network = resnet18(num_classes=10)
print(network)

## 2. 训练、验证、保存模型并绘制曲线

下面这个回调会在每个 epoch 结束时做三件事：

- 统计当前 epoch 的平均训练 loss
- 在测试集上计算 accuracy
- 若 accuracy 创下新高，则覆盖保存 `cifar10resnet18.ckpt`

如果你的老师要求“精度达标后再截图”，可以直接看每个 epoch 打印出来的验证精度。

In [ ]:
def to_scalar(value):
    if isinstance(value, (tuple, list)):
        value = value[0]
    if isinstance(value, Tensor):
        value = value.asnumpy()
    if isinstance(value, np.ndarray):
        return float(value.mean())
    return float(value)


class EvalHistory(Callback):
    def __init__(self, model, network, eval_dataset, ckpt_path, target_accuracy=0.0):
        super().__init__()
        self.model = model
        self.network = network
        self.eval_dataset = eval_dataset
        self.ckpt_path = str(ckpt_path)
        self.target_accuracy = target_accuracy
        self.train_losses = []
        self.val_accuracies = []
        self.best_accuracy = 0.0
        self.best_epoch = 0
        self._epoch_losses = []

    def on_train_epoch_begin(self, run_context):
        self._epoch_losses = []

    def on_train_step_end(self, run_context):
        cb_params = run_context.original_args()
        self._epoch_losses.append(to_scalar(cb_params.net_outputs))

    def on_train_epoch_end(self, run_context):
        cb_params = run_context.original_args()
        epoch = int(cb_params.cur_epoch_num)
        epoch_loss = float(np.mean(self._epoch_losses)) if self._epoch_losses else float("nan")
        eval_result = self.model.eval(self.eval_dataset, dataset_sink_mode=False)
        val_acc = float(eval_result["acc"])

        self.train_losses.append(epoch_loss)
        self.val_accuracies.append(val_acc)

        improved = val_acc > self.best_accuracy
        if improved:
            self.best_accuracy = val_acc
            self.best_epoch = epoch
            save_checkpoint(self.network, self.ckpt_path)

        target_msg = ""
        if self.target_accuracy and val_acc >= self.target_accuracy:
            target_msg = "  <-- target reached"

        print(
            f"Epoch {epoch:02d} | train_loss={epoch_loss:.4f} | "
            f"val_acc={val_acc:.4f} | best_acc={self.best_accuracy:.4f}{target_msg}"
        )
        if improved:
            print(f"Saved best checkpoint to: {self.ckpt_path}")


def plot_history(history_callback):
    epochs = np.arange(1, len(history_callback.train_losses) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history_callback.train_losses, marker="o")
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history_callback.val_accuracies, marker="o", color="tab:orange")
    axes[1].axhline(TARGET_ACCURACY, color="tab:red", linestyle="--", label=f"target={TARGET_ACCURACY:.2f}")
    axes[1].set_title("Validation Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    fig.tight_layout()
    save_path = OUTPUT_ROOT / "training_curves.png"
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Training curves saved to: {save_path}")

In [ ]:
network = resnet18(num_classes=10)

lr = nn.cosine_decay_lr(
    min_lr=MIN_LR,
    max_lr=MAX_LR,
    total_step=steps_per_epoch * EPOCHS,
    step_per_epoch=steps_per_epoch,
    decay_epoch=EPOCHS,
)

loss_fn = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction="mean")
optimizer = nn.Momentum(
    params=network.trainable_params(),
    learning_rate=lr,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

model = Model(network, loss_fn=loss_fn, optimizer=optimizer, metrics={"acc": nn.Accuracy()})
history = EvalHistory(
    model=model,
    network=network,
    eval_dataset=test_ds,
    ckpt_path=CKPT_PATH,
    target_accuracy=TARGET_ACCURACY,
)

model.train(
    EPOCHS,
    train_ds,
    callbacks=[TimeMonitor(), LossMonitor(per_print_times=steps_per_epoch), history],
    dataset_sink_mode=False,
)

In [ ]:
plot_history(history)
print(f"Best accuracy: {history.best_accuracy:.4f} at epoch {history.best_epoch}")
print(f"Best checkpoint exists: {CKPT_PATH.exists()} -> {CKPT_PATH}")

In [ ]:
best_net = resnet18(num_classes=10)
param_dict = load_checkpoint(str(CKPT_PATH))
load_param_into_net(best_net, param_dict)
best_net.set_train(False)

best_model = Model(best_net, metrics={"acc": nn.Accuracy()})
best_result = best_model.eval(test_ds, dataset_sink_mode=False)
print("Best checkpoint test accuracy:", best_result)

## 3. 可视化第一层 `64` 个 `7×7` 卷积核

AlexNet 2012 Fig.3 展示的是第一层学到的卷积核。  
这里直接取训练后 ResNet18 的 `conv1` 权重，按 RGB 图像方式显示 `64` 个卷积核。

In [ ]:
def normalize_kernel_image(kernel):
    kernel = kernel.transpose(1, 2, 0)
    kernel_min = kernel.min()
    kernel_max = kernel.max()
    if kernel_max - kernel_min < 1e-8:
        return np.zeros_like(kernel)
    return (kernel - kernel_min) / (kernel_max - kernel_min)


conv1_weight = best_net.conv1.conv.weight.asnumpy()
print("conv1 weight shape:", conv1_weight.shape)

fig, axes = plt.subplots(8, 8, figsize=(12, 12))
for idx, ax in enumerate(axes.flat):
    ax.imshow(normalize_kernel_image(conv1_weight[idx]))
    ax.axis("off")

fig.suptitle("ResNet18 conv1: 64 kernels of size 7x7", fontsize=14)
fig.tight_layout()
kernel_path = OUTPUT_ROOT / "resnet18_conv1_kernels.png"
fig.savefig(kernel_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Kernel figure saved to: {kernel_path}")

## 4. 复现“测试图像 + 最邻近训练图像”检索图

做法与 AlexNet 2012 Fig.4 对应：

- 使用最后隐藏层（全连接层之前）的特征向量
- 对 5 张测试图像分别计算与所有训练图像特征的欧氏距离
- 取距离最小的 6 张训练图像作为最近邻

In [ ]:
class FeatureExtractor(nn.Cell):
    def __init__(self, network):
        super().__init__()
        self.network = network

    def construct(self, x):
        return self.network.forward_features(x)


def extract_features(dataset, network):
    feature_net = FeatureExtractor(network)
    features = []
    labels = []

    for batch in dataset.create_dict_iterator():
        batch_features = feature_net(batch["image"]).asnumpy()
        batch_labels = batch["label"].asnumpy()
        features.append(batch_features.astype(np.float32))
        labels.append(batch_labels.astype(np.int32))

    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)
    return features, labels


def fetch_samples_by_indices(dataset, indices):
    indices = sorted(set(int(i) for i in indices))
    index_set = set(indices)
    samples = {}
    cursor = 0

    for batch in dataset.create_dict_iterator():
        images = batch["image"].asnumpy()
        labels = batch["label"].asnumpy()
        batch_size = images.shape[0]

        for i in range(batch_size):
            if cursor in index_set:
                samples[cursor] = {
                    "image": images[i],
                    "label": int(labels[i]),
                }
            cursor += 1

        if len(samples) == len(indices):
            break

    return samples


train_features, train_labels = extract_features(train_eval_ds, best_net)
test_features, test_labels = extract_features(test_ds, best_net)

print("train_features:", train_features.shape)
print("test_features:", test_features.shape)

In [ ]:
def select_diverse_test_indices(labels, count=5):
    selected = []
    seen = set()

    for idx, label in enumerate(labels):
        label = int(label)
        if label not in seen:
            selected.append(idx)
            seen.add(label)
        if len(selected) == count:
            break

    return selected


query_indices = select_diverse_test_indices(test_labels, count=5)
query_features = test_features[query_indices]

nearest_indices = []
for feature in query_features:
    distances = np.sum((train_features - feature[None, :]) ** 2, axis=1)
    nearest = np.argsort(distances)[:6]
    nearest_indices.append(nearest)

nearest_indices = np.array(nearest_indices)
print("Selected test indices:", query_indices)
print("Nearest neighbor index matrix shape:", nearest_indices.shape)

In [ ]:
needed_train_indices = nearest_indices.reshape(-1).tolist()
test_samples = fetch_samples_by_indices(test_ds, query_indices)
train_samples = fetch_samples_by_indices(train_eval_ds, needed_train_indices)


def show_retrieval_figure(query_indices, nearest_indices, test_samples, train_samples):
    rows = len(query_indices)
    cols = 7
    fig, axes = plt.subplots(rows, cols, figsize=(16, 3 * rows))

    if rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, query_idx in enumerate(query_indices):
        query_image = denormalize(np.expand_dims(test_samples[query_idx]["image"], axis=0))[0]
        query_label = test_samples[query_idx]["label"]

        axes[row, 0].imshow(query_image)
        axes[row, 0].set_title(f"test\n{CLASS_NAMES[query_label]}")
        axes[row, 0].axis("off")

        for col, train_idx in enumerate(nearest_indices[row], start=1):
            train_image = denormalize(np.expand_dims(train_samples[int(train_idx)]["image"], axis=0))[0]
            train_label = train_samples[int(train_idx)]["label"]
            axes[row, col].imshow(train_image)
            axes[row, col].set_title(f"nn{col}\n{CLASS_NAMES[train_label]}")
            axes[row, col].axis("off")

    fig.suptitle("Nearest training images in the last hidden layer feature space", fontsize=14)
    fig.tight_layout()
    save_path = OUTPUT_ROOT / "feature_nearest_neighbors.png"
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Nearest-neighbor figure saved to: {save_path}")


show_retrieval_figure(query_indices, nearest_indices, test_samples, train_samples)